# GK-2A 12:00~14:00 Multi-year Pipeline
2019~2025년 각 8/24~8/30에 대해 12:00~14:00 KST 10분 간격 GK-2A 16채널을 수집하고, 14:00 ASOS TA/HM 라벨과 결합합니다. 마지막에 LSTM/GRU용 long CSV와 HGB/CatBoost/LightGBM용 wide(tabular) CSV를 모두 생성합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
START_MMDD, END_MMDD = '08-24', '08-30'
START_TIME, END_TIME = '12:00', '14:00'
STEP_MINUTES = 10
BRANCH = 'agent/shortterm-12to14-pipeline'
REPO_DIR = Path('/content/SME_DATA')
OUTPUT_ROOT = Path('/content/drive/MyDrive/SME_DATA/processed_station_features/shortterm_12to14_data')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('YEARS:', YEARS)
print('OUTPUT:', OUTPUT_ROOT)


In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'
if not REPO_DIR.exists():
    subprocess.run(['git','clone','-b',BRANCH,REPO_URL,str(REPO_DIR)], check=True)
else:
    os.chdir(REPO_DIR)
    subprocess.run(['git','fetch','origin'], check=True)
    subprocess.run(['git','checkout',BRANCH], check=True)
    subprocess.run(['git','pull','origin',BRANCH], check=True)
os.chdir(REPO_DIR)
!pip -q install -e .
!pip -q install requests pandas numpy xarray h5netcdf netCDF4 pyyaml


In [ ]:
import os
from getpass import getpass
try:
    from google.colab import userdata
    key = userdata.get('KMA_API_KEY')
except Exception:
    key = None
if not key:
    key = getpass('KMA_API_KEY 입력: ').strip()
if not key:
    raise ValueError('KMA_API_KEY가 비어 있습니다.')
os.environ['KMA_API_KEY'] = key
print('KMA_API_KEY 설정 완료')


## Multi-year 수집 + tabular 변환 + TA/HM 병합
이 셀 하나가 2019~2025 각 연도를 순서대로 처리합니다. 이미 받은 정상 파일은 자동으로 건너뜁니다. 중간에 끊기면 같은 셀을 다시 실행하면 됩니다.


In [ ]:
import sys, subprocess
cmd = [
    sys.executable, 'scripts/collect_shortterm_years.py',
    '--years', *[str(y) for y in YEARS],
    '--start-mmdd', START_MMDD, '--end-mmdd', END_MMDD,
    '--start-time', START_TIME, '--end-time', END_TIME,
    '--step-minutes', str(STEP_MINUTES),
    '--output-root', str(OUTPUT_ROOT),
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


## 결과 검수


In [ ]:
import pandas as pd
from pandas.errors import EmptyDataError
COMBINED_DIR = OUTPUT_ROOT / 'datasets' / 'combined'
SUMMARY = COMBINED_DIR / 'shortterm_multiyear_summary.csv'
LONG = COMBINED_DIR / 'shortterm_long_2019to2025.csv'
WIDE = COMBINED_DIR / 'shortterm_wide_2019to2025.csv'
LABELS = COMBINED_DIR / 'shortterm_labels_1400_2019to2025.csv'
FAIL = COMBINED_DIR / 'shortterm_collection_failures_2019to2025.csv'
BUILD_MISSING = COMBINED_DIR / 'shortterm_build_missing_2019to2025.csv'
summary = pd.read_csv(SUMMARY)
display(summary)
long_df = pd.read_csv(LONG)
wide_df = pd.read_csv(WIDE)
labels_df = pd.read_csv(LABELS)
print('LONG:', long_df.shape, 'WIDE:', wide_df.shape, 'LABELS:', labels_df.shape)
print('Years:', sorted(wide_df['Year'].unique().tolist()))
print('Stations:', wide_df['STN_ID'].nunique())
print('Long timesteps:', sorted(long_df['TimeKST'].unique().tolist()))
print('TA missing:', labels_df['TA'].isna().sum(), 'HM missing:', labels_df['HM'].isna().sum())
for path, name in [(FAIL,'collection failures'), (BUILD_MISSING,'build issues')]:
    try:
        df = pd.read_csv(path) if path.exists() and path.stat().st_size else pd.DataFrame()
    except EmptyDataError:
        df = pd.DataFrame()
    print(name + ':', len(df))


In [ ]:
train = wide_df[wide_df['Year'] <= 2023].copy()
valid = wide_df[wide_df['Year'] == 2024].copy()
test = wide_df[wide_df['Year'] == 2025].copy()
print('Train 2019~2023:', train.shape)
print('Valid 2024:', valid.shape)
print('Test 2025:', test.shape)
print('최종 tabular:', WIDE)
print('최종 LSTM/GRU long:', LONG)
